# 01 — Introdução e taxonomia

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 01/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/01_introducao_taxonomia.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_01_introducao_taxonomia.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Distinguir recomendação por conteúdo, colaborativa e popularidade
- Entender feedback explícito e implícito e objetivos de produto

## Fundamentação conceitual
Um sistema de recomendação estima a relevância de um item $i$ para um usuário $u$. Modelos por conteúdo utilizam atributos dos itens; filtragem colaborativa utiliza padrões de interação. Popularidade é um baseline importante. **Ausência de avaliação não implica nota zero.**

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Primeiro contato com MovieLens

In [2]:
print(movies.head(4).to_string(index=False))
print(ratings.rating.describe())
assert ratings.rating.between(0.5,5).all()

 movieId                    title                                      genres
       1         Toy Story (1995) Adventure|Animation|Children|Comedy|Fantasy
       2           Jumanji (1995)                  Adventure|Children|Fantasy
       3  Grumpier Old Men (1995)                              Comedy|Romance
       4 Waiting to Exhale (1995)                        Comedy|Drama|Romance
count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64


**Interpretação.** O intervalo de notas revela feedback explícito, não cliques.

## Interações e cobertura

In [3]:
n_users=ratings.userId.nunique(); n_items=ratings.movieId.nunique()
density=len(ratings)/(n_users*n_items)
print({'usuarios':n_users,'itens_avaliados':n_items,'densidade':round(density,5),'esparsidade':round(1-density,5)})

{'usuarios': 610, 'itens_avaliados': 9724, 'densidade': 0.017, 'esparsidade': 0.983}


**Interpretação.** A matriz é esparsa; esta característica orienta a escolha das técnicas.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** objetivo da recomendação. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **objetivo da recomendação** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **objetivo da recomendação** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Taxonomia e critérios de avaliação

Uma recomendação pode otimizar previsão de nota, ordenação, descoberta ou satisfação. O **objetivo** define o conjunto de candidatos, o sinal de relevância e a métrica. Popularidade é um baseline não personalizado; filtragem por conteúdo compara atributos; filtragem colaborativa usa padrões coletivos de interação. Nunca interprete a ausência de nota como rejeição.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [4]:
# Contrastar cobertura do catálogo com densidade da matriz observada
cobertura = ratings.movieId.nunique() / movies.movieId.nunique()
densidade = len(ratings) / (ratings.userId.nunique() * ratings.movieId.nunique())
assert 0 <= cobertura <= 1 and 0 <= densidade <= 1
print(f'Cobertura dos itens do catálogo: {cobertura:.3%}')
print(f'Densidade entre usuários e itens observados: {densidade:.3%}')

Cobertura dos itens do catálogo: 99.815%
Densidade entre usuários e itens observados: 1.700%


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
MovieLens reflete preferências de quem avaliou; não permite inferir exposição a todos os filmes.

## Exercícios
1. Calcule a proporção de filmes do catálogo que receberam alguma nota.
2. Explique por que uma recomendação popular não é personalizada.

## Soluções comentadas
`ratings.movieId.nunique()/movies.movieId.nunique()` mede cobertura observada. A popularidade ignora histórico individual.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [5]:
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
resumo = pd.DataFrame([{
    "usuarios_ativos": ratings.userId.nunique(),
    "itens_avaliados": ratings.movieId.nunique(),
    "itens_catalogo": movies.movieId.nunique(),
    "avaliacoes": len(ratings),
    "densidade_ativos": len(ratings)/(ratings.userId.nunique()*ratings.movieId.nunique()),
    "cobertura_catalogo": ratings.movieId.nunique()/movies.movieId.nunique(),
}])
resumo.to_csv(out / "01_resumo_movielens.csv", index=False)
print('Outputs do capítulo 01 salvos em:', out.resolve())


Outputs do capítulo 01 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
